# Machine-Learning-Aufgabe: Herzerkrankungen aus PostgreSQL

Diese Musterlösung behält den Workflow mit der Datenbank bei: fünf Tabellen untersuchen und verknüpfen,
die Zielvariable korrigieren, ohne die Quelle zu ändern, und Klassifikatoren vergleichen.

Führ die Zellen der Reihe nach aus, nachdem du `.env` eingerichtet hast. Jeder Datenbankzugriff ist schreibgeschützt.
Das Modell klassifiziert dokumentierte Erkrankungen in historischen Lehrdaten, keine künftigen
Herzinfarkte. Bearbeite zuerst die Aufgabe und vergleich dann deine Entscheidungen.

## Schritt 1: Ziel und Forschungsfrage festlegen

**Frage:** Können die 13 erfassten diagnostischen Benchmark-Attribute
Einträge mit Krankheit von Einträgen ohne Krankheit unterscheiden?

1. Verknüpf die fünf Tabellen im Schema `heart` und nutz die Patienten-IDs als Schlüssel.
2. Prüf die Abdeckung des Joins, doppelte Profile, Kategoriecodes und fehlende Werte.
3. Leite `disease_present = 1 - heart_attack` ab; Klasse 1 bedeutet Krankheit vorhanden.
4. Teil die Daten vor der EDA auf, fitte dann in einer Pipeline die Vorverarbeitung und trainiere die logistische Regression.
5. Tune die Hyperparameter auf den Trainings-Folds, vergleich die Kandidaten auf den Validierungsdaten und evaluiere
   den festgeschriebenen Ansatz einmal auf den Testdaten.

Wir nehmen an, dass die ausgewählten Messwerte zum Zeitpunkt der Vorhersage bereits vorliegen.
Für ein Tool, das vor den diagnostischen Tests eingesetzt wird, träfe diese Annahme nicht zu.

### Erfolgsmetrik

Wähl in dieser Aufgabe Modelle nach **F2 für Krankheit vorhanden (`pos_label=1`)** aus
und gib außerdem Precision, Recall und Accuracy an. Das ist eine didaktische Entscheidung,
die übersehene Einträge mit Krankheit betont; sie ist keine klinische Anforderung.

| Ergebnis | Interpretation |
|---|---|
| True Positive (TP) | Krankheit vorhanden, als vorhanden vorhergesagt |
| False Negative (FN) | Krankheit vorhanden, als nicht vorhanden vorhergesagt |
| False Positive (FP) | Keine Krankheit dokumentiert, als vorhanden vorhergesagt |
| True Negative (TN) | Keine Krankheit dokumentiert, als nicht vorhanden vorhergesagt |

- **Recall** = TP / (TP + FN): Wie viele dokumentierte Krankheitsfälle werden gefunden?
- **Precision** = TP / (TP + FP): Wie viele positive Vorhersagen sind korrekt?
- **Accuracy** = (TP + TN) / alle Einträge: Wie viele Vorhersagen sind insgesamt korrekt?

$$
F_2 = \frac{5\,TP}{5\,TP + FP + 4\,FN}
$$

F2 betont den Recall stärker als F1. Der Koeffizient 4 bei FN kommt von
`beta=2`; er belegt **nicht**, dass ein übersehener Fall in der Praxis genau viermal
so viel kostet wie ein Fehlalarm. Ein Klassifikator, der alle als positiv einstuft,
hat einen perfekten Recall, aber viele False Positives; ein hoher Recall allein reicht also nicht.
Nutz in allen Vergleichen dieselbe Metrik und dieselbe Klassendefinition.

Siehe die [Definition des F-beta-Scores in scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.fbeta_score.html).

### Was du verwenden und beachten solltest

- **Skaliere deine Daten:** Vergleich unskalierte, standardisierte und Min-Max-skalierte Pipelines.
- **Tune dein Modell:** Nutz überschaubare Randomized Searches und Grid Searches mit denselben CV-Folds.
- **Halt die Metrik einheitlich:** Behandle Krankheit vorhanden durchgängig als Klasse 1.

### Datenüberblick

Das ist eine **Lehrfassung des Datensatzes UCI Cleveland Heart Disease**,
aufgeteilt auf fünf PostgreSQL-Tabellen. Eine verknüpfte Zeile steht für einen Patienteneintrag.
Die Aufgabe ist die retrospektive Klassifikation einer **dokumentierten Erkrankung**, nicht die Vorhersage
eines künftigen Herzinfarkts. Der Datensatz ist klein und historisch; das ist kein Tool für medizinische
Entscheidungen. [Originaldokumentation von UCI](https://archive.ics.uci.edu/dataset/45/heart+disease).

#### Zielvariable: dem Spaltennamen nicht trauen

Das Feld `heart_attack` der Datenbank hat eine **umgekehrte binäre Kodierung** im Vergleich
zum ursprünglichen UCI-Label für eine vorhandene Krankheit:

| `heart_attack` in der Datenbank | Abgeleitetes `disease_present` | Bedeutung |
|---|---|---|
| 1 | 0 | Laut Definition der Quelle keine Krankheit dokumentiert |
| 0 | 1 | Krankheit vorhanden |

Behalt das ursprüngliche Feld bei, prüf, dass es nur 0 und 1 enthält, und leite
`disease_present = 1 - heart_attack` ab. Nutz **nur `disease_present` als
Zielvariable für die Modellierung**. Klasse 0 bedeutet nicht allgemein gesund.
Keine der beiden Zielspalten gehört zu den Features.

Diese Zuordnung wurde am 2026-10-08 mit UCI abgeglichen: Alle 303 Einträge mit den
nötigen verknüpften Tabellen stimmen nach Berücksichtigung der umkodierten Kategorien und fehlender
Messwerte überein. Eine höhere Herzfrequenz oder weniger Angina in Klasse 1 der Datenbank
deutet deshalb nicht auf einen umgekehrten klinischen Zusammenhang hin: Diese Klasse bedeutet
**keine Krankheit dokumentiert**. Nutz die dokumentierte Kodierung, nicht Korrelationen,
um die Labels zu interpretieren.

#### Feature-Wörterbuch: Datenbankcodes, nicht die ursprünglichen UCI-Codes

Ein EKG zeichnet die elektrische Aktivität des Herzens auf. Angina bezeichnet Brustschmerzen,
die mit einer verminderten Durchblutung des Herzens einhergehen; die ST-Strecke ist ein Teil
des EKGs. Die numerischen Codes unten sind Kennungen von Kategorien, keine Messwerte.

| Datenbankspalte | Bedeutung und Kodierung | Behandlung |
|---|---|---|
| `patient_id` | Patientenschlüssel aus dem Join | Auf Eindeutigkeit prüfen; von den Eingaben ausschließen |
| `age` | Alter in Jahren | Numerisch |
| `sex` | Erfasstes Geschlecht: 0 = weiblich, 1 = männlich | Kategorial |
| `chest_pain_type` | 0 = asymptomatisch, 1 = atypische Angina, 2 = nicht anginöser Schmerz, 3 = typische Angina | Kategorial |
| `resting_blood_pressure` | Ruheblutdruck bei der Aufnahme, mm Hg | Numerisch |
| `serum_cholestoral` | Serumcholesterin, mg/dL; die Schreibweise folgt dem Schema | Numerisch |
| `fasting_blood_sugar` | Über 120 mg/dL: 0 = falsch, 1 = wahr | Kategorial |
| `resting_electrocardiographic_results` | 0 = wahrscheinliche/sichere linksventrikuläre Hypertrophie, 1 = normal, 2 = ST-T-Veränderung | Kategorial |
| `maximum_heartrate_achieved` | Maximale Herzfrequenz unter Belastung, Schläge/Minute | Numerisch |
| `exercise_induced_angina` | 0 = nein, 1 = ja | Kategorial |
| `oldpeak` | Belastungsinduzierte ST-Senkung im Vergleich zur Ruhe | Numerisch |
| `slope_of_the_peak_exercise_st_segment` | 0 = absteigend, 1 = flach, 2 = ansteigend | Kategorial |
| `number_of_major_vessels_colored_by_flourosopy` | Gefäßanzahl 0 bis 3; Code 4 steht für fehlend | Numerische Anzahl; 4 durch fehlend ersetzen |
| `thal` | Testergebnis: 1 = fixierter Defekt, 2 = normal, 3 = reversibler Defekt; Code 0 steht für fehlend | Kategorial; 0 durch fehlend ersetzen |
| `real_data` | Herkunftsmarkierung der Lehrdaten, manchmal nicht vorhanden | Nur für die Qualitätsprüfung; von den Eingaben ausschließen |
| `heart_attack` | Ursprüngliches, invertiertes binäres Feld | Zur Kontrolle beibehalten; von den Eingaben ausschließen |
| `disease_present` | Abgeleitetes Label: 0 = keine Krankheit dokumentiert, 1 = Krankheit vorhanden | Nur Zielvariable |

Die Datenbankcodes unterscheiden sich von den ursprünglichen UCI-Codes; **misch die beiden
Wörterbücher nicht**. Behalt die Kategorien der Datenbank unverändert bei und kodiere sie
per One-Hot Encoding. Wandle die beiden Platzhaltercodes für fehlende Werte vor der Imputation um.

Warum diese 13 Eingaben? Damit bleiben die diagnostischen Benchmark-Attribute der Quelle erhalten.
Die Gefäßanzahl und `thal` hängen von speziellen Tests ab, und mehrere andere Features
stammen aus einem Belastungstest. Bei einem früheren Screening lägen sie nicht alle
vor. Eingaben auszuwählen heißt zu wissen, was zum Zeitpunkt der Vorhersage
verfügbar ist, und nicht einfach jede Spalte der Datenbank zu verwenden.

#### Was ist anders als in der Originaldatei?

Die Überprüfung ergab 303 verknüpfte Einträge: 264 stimmen nach der Umkodierung mit allen
13 UCI-Messwerten überein, 39 stimmen mit den verfügbaren Messwerten überein, haben aber zusätzlich
fehlende Werte bei Alter, Geschlecht oder maximaler Herzfrequenz. Ein ursprüngliches Profil aus Eingaben und Label
kommt unter zwei IDs doppelt vor, während ein anderer ursprünglicher Eintrag fehlt.

Der frühere Left Join behielt außerdem 50 zusätzliche Patientinnen und Patienten ohne die nötigen
Einträge in den Messwerttabellen. Unterscheide **einen fehlenden verknüpften Eintrag** von
**einem fehlenden Messwert in einem vorhandenen Eintrag**: Schließ den ersten Fall aus, behalt aber
den zweiten für die auf den Trainingsdaten gefittete Imputation. Nach dem Entfernen des wiederholten Profils
verwendet die aktuelle Lösung 302 Einträge. Diese Zahlen beschreiben die überprüfte
Lehrdatenbank, keinen garantierten künftigen Zustand der Datenbank.

## Schritt 2: Daten beschaffen

Wir importieren die benötigten Bibliotheken und laden dann die Herzdaten aus der Datenbank herunter.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv
from scipy.stats import loguniform, uniform
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    fbeta_score,
    make_scorer,
    precision_score,
    recall_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler
from sqlalchemy import URL, create_engine, text

plt.rcParams.update(
    {"figure.figsize": (8, 5), "axes.facecolor": "white", "axes.edgecolor": "black"}
)
pd.options.display.float_format = "{:,.2f}".format
RSEED = 42

### Aus der Datenbank laden

Untersuch die fünf `heart`-Tabellen in DBeaver. Die Abfrage unten ist der Referenz-Join,
kein Ersatz für eine CSV-Datei. Sie wählt die Felder explizit aus und gibt dem
Patientenschlüssel den Alias `patient_id`.

Mit einem **Inner Join** setzen wir in jeder Messwerttabelle eine vorhandene Zeile voraus.
Das entfernt Patientinnen und Patienten ohne verknüpfte Messwerte, behält aber fehlende
Messwerte innerhalb vorhandener Zeilen. Die eindeutig erfundene Herkunftsmarkierung wird
ausgeschlossen; Einträge, denen nur die Markierung fehlt, bleiben erhalten.

Kopiere `.env.example` nach `.env` und trag die bereitgestellten Zugangsdaten ein.
Die Verbindungs-URL wird automatisch aus den Feldern gebaut; Passwörter brauchen keine
manuelle URL-Kodierung. Vorhandene vollständige `DB_STRING`-Werte werden unterstützt.
Die Transaktion ist schreibgeschützt; Zugangsdaten und private Exporte bleiben lokal.

In [ ]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "solutions":
    PROJECT_ROOT = PROJECT_ROOT.parent
load_dotenv(PROJECT_ROOT / ".env")

DB_STRING = os.getenv("DB_STRING")
if not DB_STRING or "<" in DB_STRING:
    # URL aus den Feldern in .env bauen; URL.create behandelt Sonderzeichen.
    field_names = ("USER_DB", "PASSWORD", "HOST", "DATABASE")
    fields = {name: os.getenv(name) for name in field_names}
    if any(not value or "<" in value for value in fields.values()):
        raise ValueError(
            "Set DB_STRING or USER_DB, PASSWORD, HOST, DATABASE and PORT in .env."
        )
    try:
        port = int(os.getenv("PORT") or "5432")
    except ValueError:
        raise ValueError("PORT in .env must be an integer.") from None
    DB_STRING = URL.create(
        "postgresql+psycopg2",
        username=fields["USER_DB"],
        password=fields["PASSWORD"],
        host=fields["HOST"],
        port=port,
        database=fields["DATABASE"],
    )

db = create_engine(DB_STRING, connect_args={"connect_timeout": 10})

In [ ]:
# Inner Joins setzen die Messwerteinträge voraus; einzelne fehlende Werte bleiben erhalten.
query_string = """
SELECT p.id AS patient_id, p.age, p.sex,
       cp.chest_pain_type,
       bm.serum_cholestoral, bm.fasting_blood_sugar, bm.thal,
       pva.resting_blood_pressure, pva.resting_electrocardiographic_results,
       pva.maximum_heartrate_achieved, pva.exercise_induced_angina, pva.oldpeak,
       pva.slope_of_the_peak_exercise_st_segment,
       pva.number_of_major_vessels_colored_by_flourosopy, pva.real_data,
       hah.heart_attack
FROM heart.patient AS p
JOIN heart.chest_pain AS cp ON p.id = cp.patient_id
JOIN heart.blood_metrics AS bm ON p.id = bm.patient_id
JOIN heart.pressure_vessels_angina AS pva ON p.id = pva.patient_id
JOIN heart.heart_attack_history AS hah ON p.id = hah.patient_id
WHERE pva.real_data <> 'Evgeny likes white wine for lunch and red wine for dinner'
   OR pva.real_data IS NULL
ORDER BY p.id
"""
# Die Datenbank bleibt unverändert; die Verbindung auch dann schließen, wenn das Lesen fehlschlägt.
try:
    with db.connect() as connection, connection.begin():
        connection.execute(text("SET TRANSACTION READ ONLY"))
        df_sqlalchemy = pd.read_sql(text(query_string), connection)
finally:
    db.dispose()
df_sqlalchemy.head()

In [ ]:
# Optionaler Export: das ignorierte Datenverzeichnis nur bei Bedarf anlegen.
# (PROJECT_ROOT / "data").mkdir(exist_ok=True)

In [ ]:
# Optional: den fertigen Join lokal speichern, nie in einer versionierten Datei.
# df_sqlalchemy.to_csv(PROJECT_ROOT / "data" / "heart_data.csv", index=False)

In [ ]:
# Direkt mit dem Abfrageergebnis weitermachen; ein Umweg über CSV ist nicht nötig.
df = df_sqlalchemy.copy()

In [ ]:
df.head()

In [ ]:
print(f"Eligible observations: {df.shape[0]}")
print(f"Columns (including identifier, provenance and target): {df.shape[1]}")

### Strukturelle Bereinigung vor der Aufteilung

Behalt die vorhandenen kurzen Spaltenaliasse bei, die unten verwendet werden. Prüf, ob es pro verknüpfter Zeile einen nicht fehlenden
Patientenschlüssel gibt; widersprüchliche Zeilen brauchen eine Korrektur des Joins, keine
stillschweigende Deduplizierung.

Validiere die rohe Zielvariable, behalt sie bei und leite `disease_present` so ab, dass
Klasse 1 Krankheit bedeutet. Wandle bekannte Platzhaltercodes für fehlende Werte in `NaN` um;
dafür dient das Wörterbuch, keine gelernten Kennzahlen.

Entfern exakt gleiche Profile aus Eingaben und Label vor der Aufteilung und ignoriere dabei die ID und
die Herkunftsmarkierung. Die Überprüfung fand ein wiederholtes UCI-Profil unter zwei Datenbank-IDs;
das Entfernen dieses Profils verhindert, dass exakt gleiche Beobachtungen in mehreren Teilmengen landen.
Es beweist nicht, dass die Einträge dieselbe Person beschreiben. Gewöhnliche fehlende
Messwerte bleiben für die auf den Trainingsdaten gefittete Imputation erhalten.

In [ ]:
# Spalten umbenennen

df.rename(
    columns={
        "fasting_blood_sugar": "fasting_bs",
        "resting_blood_pressure": "resting__bp",
        "resting_electrocardiographic_results": "electrocardiography",
        "maximum_heartrate_achieved": "max_heartrate",
        "exercise_induced_angina": "exercise_ind_angina",
        "slope_of_the_peak_exercise_st_segment": "slope_of_peak_exercise",
        "number_of_major_vessels_colored_by_flourosopy": "major_vessels_colored",
    },
    inplace=True,
)

In [ ]:
# Integrität des Joins prüfen, bevor stillschweigend Einträge entfernt werden.
if df["patient_id"].isna().any() or df["patient_id"].duplicated().any():
    raise ValueError("Expected one nonmissing patient_id per row. Fix the join first.")
if not df["heart_attack"].dropna().isin([0, 1]).all():
    raise ValueError(
        "The raw database target must contain only the documented codes 0 and 1."
    )
print(f"Missing target labels: {int(df['heart_attack'].isna().sum())}")

# Die Kategoriecodes der Datenbank unterscheiden sich von den ursprünglichen UCI-Codes.
allowed_codes = {
    "sex": {0, 1},
    "chest_pain_type": {0, 1, 2, 3},
    "fasting_bs": {0, 1},
    "electrocardiography": {0, 1, 2},
    "exercise_ind_angina": {0, 1},
    "slope_of_peak_exercise": {0, 1, 2},
    "major_vessels_colored": {0, 1, 2, 3, 4},
    "thal": {0, 1, 2, 3},
}
for column, allowed in allowed_codes.items():
    unexpected = set(df[column].dropna().unique()) - allowed
    if unexpected:
        raise ValueError(f"Unexpected codes in {column}: {unexpected}")

In [ ]:
# Unbekannte Labels sind für überwachtes Lernen unbrauchbar; Ausschlüsse melden.
before = len(df)
df = df.dropna(subset=["heart_attack"]).copy()
print(f"Excluded {before - len(df)} records without a target.")

# Rohes Label beibehalten; seine Bedeutung in einer neuen Zielvariable für die Modellierung korrigieren.
df["heart_attack"] = df["heart_attack"].astype(int)
df["disease_present"] = 1 - df["heart_attack"]

# Platzhaltercodes für fehlende Werte dekodieren, ohne Ersatzwerte aus den Daten zu lernen.
df["major_vessels_colored"] = df["major_vessels_colored"].replace(4, np.nan)
df["thal"] = df["thal"].replace(0, np.nan)

# Beim Prüfen auf identische Profile aus Eingaben und Label IDs und Herkunftsmarkierungen ignorieren.
profile_cols = [
    column
    for column in df.columns
    if column not in ("patient_id", "real_data", "heart_attack")
]
repeated_profiles = df.duplicated(subset=profile_cols)
print(f"Repeated input-and-label profiles removed: {int(repeated_profiles.sum())}")
df = df.loc[~repeated_profiles].copy()
if set(df["disease_present"].unique()) != {0, 1}:
    raise ValueError(
        "This exercise requires both disease-present and disease-absent records."
    )
print(f"Retained {len(df)} records, including individual missing measurements.")
pd.crosstab(df["heart_attack"], df["disease_present"])

## Schritt 3: Aufteilung in Trainings-, Validierungs- und Testdaten

Leg 20% für den abschließenden Test beiseite. Teil den Rest in Trainings- und Validierungsdaten auf (60% / 20% der gesamten Stichprobe). Die Stratifizierung erhält die Klassenanteile ungefähr. Das setzt unabhängige Patientinnen und Patienten aus derselben Grundgesamtheit der Quelle voraus.

EDA und CV-Suchen nutzen nur die Trainingsdaten. Mit dem Validierungsdatensatz werden Kandidaten verglichen; der Testdatensatz bleibt bis Schritt 9 unberührt.

Die zweite Aufteilung nimmt 25% der verbleibenden 80%; das ergibt insgesamt etwa 60% Training, 20% Validierung und 20% Test.

<img src="../assets/heart-evaluation-protocol.svg" width="650" alt="Aufgabe zu Herzerkrankungen: Training mit interner Kreuzvalidierung, separate Validierung für die Auswahl der Kandidaten, erneutes Training auf den Entwicklungsdaten und eine abschließende Evaluation auf den Testdaten.">

Anders als das Titanic-Notebook nutzt diese Lösung nach der Kreuzvalidierung auf den Trainingsdaten einen separaten Validierungsdatensatz. Beide Ansätze sind gültig; die gemeinsame Regel lautet, dass die Testdaten nie das Modell auswählen. Die Prozentangaben unten beschreiben diese Aufgabe, keine allgemeingültige Vorgabe.

In [ ]:
development_df, test_df = train_test_split(
    df, test_size=0.2, random_state=RSEED, stratify=df["disease_present"]
)
train_df, val_df = train_test_split(
    development_df,
    test_size=0.25,
    random_state=RSEED,
    stratify=development_df["disease_present"],
)
print("Training / validation / test:", len(train_df), len(val_df), len(test_df))

## Schritt 4: Datenexploration

Die gesamte Exploration unten läuft auf `train_df`. Wenn du stattdessen Features und Zielvariable getrennt aufgeteilt hast, kannst du für die EDA mit `pd.concat([X_train, y_train], axis=1)` wieder einen einzelnen DataFrame bauen.

### Grundlagen verstehen

In [ ]:
# Datentypen und Anzahl nicht fehlender Werte
train_df.info()

In [ ]:
# Deskriptive Kennzahlen
train_df.describe().T

In [ ]:
# Eindeutige Werte pro Spalte
train_df.nunique()

### Auf fehlende Werte prüfen

In [ ]:
# Anteil fehlender Werte pro Spalte
train_df.isnull().mean()

In [ ]:
# Klassenverteilung der Zielvariable
train_df["disease_present"].value_counts()

### Daten visualisieren

In [ ]:
# Alle prädiktiven Features und die Zielvariable darstellen, ohne Kennungen und Herkunft.
features = [
    c for c in train_df.columns if c not in ("patient_id", "real_data", "heart_attack")
]
nrows = (len(features) + 2) // 3
fig, axes = plt.subplots(nrows, 3, figsize=(15, 3 * nrows), squeeze=False)
for item, ax in zip(features, axes.flat):
    # Häufigkeiten bleiben für Kategoriecodes aussagekräftig; eine Dichtekurve würde Stetigkeit unterstellen.
    sns.histplot(train_df[item], ax=ax, color="#33658A").set(title=item, xlabel="")
for ax in list(axes.flat)[len(features) :]:
    ax.set_visible(False)
fig.tight_layout()

Lies die Klassenhäufigkeiten oben ab und berechne die Anteile. Nimm nicht an, dass die Live-Datenbank balanciert ist oder dieselbe Verteilung hat wie bei einem früheren Durchlauf.

### Ausreißer erkennen

In [ ]:
# Ausreißer in stetigen Messwerten untersuchen, nicht in willkürlichen Kategoriecodes.
continuous_features = [
    "age",
    "serum_cholestoral",
    "resting__bp",
    "max_heartrate",
    "oldpeak",
]
train_df[continuous_features].plot(
    kind="box", subplots=True, layout=(2, 3), figsize=(12, 7)
)
plt.tight_layout()
plt.show()

### Fehlende Messwerte und Herkunftsmarkierungen

Im Vergleich zur Quelle enthält die Datenbank zusätzliche fehlende Werte bei Alter, Geschlecht
und maximaler Herzfrequenz. Behandle sie als fehlende Messwerte, nicht als neue Kategorien.
Behalt die ursprünglichen Markierungen in `real_data` und nutz eine Kopie für die Darstellung, um fehlende Markierungen zu zeigen.

Die Markierungen reichten nicht aus, um die Identität der Quelle festzustellen: Der Abgleich mit UCI
prüfte stattdessen die verfügbaren Messwerte und die Zielvariable. Ähnlich aussehende
Verteilungen allein beweisen keine gemeinsame Herkunft.

In [ ]:
# Fehlende Features untersuchen, ohne die ursprünglichen Herkunftsmarkierungen zu überschreiben.
print(train_df[continuous_features].isna().mean().rename("missing_fraction"))
provenance_plot = train_df.copy()
provenance_plot["source_label"] = train_df["real_data"].fillna("Missing source tag")

In [ ]:
# Herkunftsgruppen anhand der Messwerte vergleichen, nicht anhand von IDs oder willkürlichen kategorialen Codes.
sns.pairplot(
    provenance_plot, vars=continuous_features, hue="source_label", corner=True, height=2
);

Fehlende Herkunftsmarkierungen beweisen nicht, dass ein Eintrag erfunden ist.
Der Abgleich mit der Quelle ergab, dass alle 303 vollständig verknüpften Einträge nach der Umkodierung
bei ihren verfügbaren Messwerten mit einem ursprünglichen UCI-Profil übereinstimmen.
Das wiederholte Profil wird vor der Aufteilung entfernt.
Die Imputer behandeln die übrigen fehlenden Eingaben nur anhand der Trainingsdaten.

Untersuch die maximale Herzfrequenz unter Belastung und andere Messwerte nach den korrigierten
Klassen von `disease_present`: 0 = keine Krankheit dokumentiert, 1 = Krankheit vorhanden.
Die alte Klasse 1 der rohen Datenbank bedeutete keine Krankheit; übernimm diese Interpretation nicht
für die korrigierte Zielvariable. Diese Plots sind explorativ, kein kausaler Beleg.

In [ ]:
# Die Messwerte anhand der korrigierten Bedeutung der Klassen vergleichen.
class_summary = train_df.groupby("disease_present").agg(
    records=("patient_id", "size"),
    mean_max_heart_rate=("max_heartrate", "mean"),
    exercise_angina_pct=("exercise_ind_angina", lambda values: 100 * values.mean()),
)
class_summary.rename(index={0: "No disease recorded", 1: "Disease present"})

In [ ]:
# Den grafischen Vergleich nach Klasse der Zielvariable sowie die numerische Zusammenfassung behalten.
sns.pairplot(
    train_df, vars=continuous_features, hue="disease_present", corner=True, height=2
)
plt.show()
train_df.groupby("disease_present")[continuous_features].median()

Beschreib die Zusammenhänge mit den Klassen, die in dieser Trainingsstichprobe tatsächlich sichtbar sind.
Sie belegen kein künftiges Herzinfarktrisiko, und ein einzelner Messwert
bestimmt nicht die Diagnose.

### Korrelationsanalyse

In [ ]:
# Heatmap der Korrelationen

fig, ax = plt.subplots(figsize=(12, 8))

# Messwerte und die binäre Zielvariable auswählen; IDs und Kategoriecodes ausschließen
df_numeric = train_df[continuous_features + ["disease_present"]]

# Korrelationen berechnen
correlations = df_numeric.corr()

# Maske für das obere Dreieck erzeugen
mask = np.zeros_like(correlations)
mask[np.triu_indices_from(mask)] = True

# Eigene divergierende Farbskala erzeugen
cmap = sns.diverging_palette(140, 10, as_cmap=True)

# Heatmap mit der Maske und dem richtigen Seitenverhältnis zeichnen

sns.heatmap(
    correlations,
    mask=mask,
    cmap=cmap,
    vmin=-1,
    vmax=1,
    center=0,
    annot=True,
    linewidths=1,
    cbar_kws={"shrink": 0.7},
    ax=ax,
);

Diese Heatmap zeigt Pearson-Zusammenhänge in den Trainingsdaten, keine kausalen Effekte.
Die korrigierte binäre Zielvariable kann einbezogen werden, aber willkürliche Kategoriecodes,
IDs, Herkunftsmarkierungen und das rohe invertierte Label sind keine stetigen Prädiktoren.

## Schritt 5: Feature Engineering

Behalt einzelne fehlende Eingaben für die Imputation innerhalb der Pipeline.
Schließ `patient_id`, `real_data`, **das rohe Feld `heart_attack`** und die
abgeleitete Zielvariable `disease_present` von den Features aus. Das rohe Feld
verrät nach der Invertierung direkt die Antwort und würde zu einem Datenleck über die Zielvariable führen.

Numerische Eingaben werden per Median imputiert; kategoriale Eingaben erhalten ihre
häufigste Kategorie in den Trainingsdaten. Die bekannten Platzhaltercodes sind bereits dekodiert,
die Ersatzwerte werden aber nur innerhalb jedes Trainings-Folds gelernt.
`keep_empty_features=True` behält eine Spalte, die in einem Fold komplett fehlt;
untersuch so einen Fall, statt ihren Fallback-Wert als Messwert zu behandeln.

Wir behalten alle 13 diagnostischen Benchmark-Eingaben. Messwerte aus speziellen Tests
wie die Gefäßanzahl und `thal` wären bei einem frühen Screening nicht alle verfügbar.

In [ ]:
# Fehlende Eingaben behalten: Die Pipeline unten lernt die Ersatzwerte.
print("Training / validation / test rows:", len(train_df), len(val_df), len(test_df))

### Features und Zielvariable festlegen

In [ ]:
# Spalten nach Behandlung gruppieren; Kennungen und Herkunft gehen nicht in die Vorhersagen ein.
num_cols = [
    "age",
    "serum_cholestoral",
    "resting__bp",
    "max_heartrate",
    "oldpeak",
    "major_vessels_colored",
]
cat_cols = [
    "sex",
    "fasting_bs",
    "exercise_ind_angina",
    "chest_pain_type",
    "thal",
    "electrocardiography",
    "slope_of_peak_exercise",
]
feature_cols = num_cols + cat_cols
assert "heart_attack" not in feature_cols and "disease_present" not in feature_cols

X_train, y_train = train_df[feature_cols], train_df["disease_present"]
X_val, y_val = val_df[feature_cols], val_df["disease_present"]

# Nur für die absichtlich fehlerhafte Demonstration mit der Kennung.
X_train_id = train_df[feature_cols + ["patient_id"]]
X_val_id = val_df[feature_cols + ["patient_id"]]
print("Training / validation shapes:", X_train.shape, X_val.shape)

### Feature-Engineering-Pipeline aufbauen

Wie im ersten Notebook kommt das gesamte Feature Engineering in einen einzigen `ColumnTransformer`, der die numerischen Spalten imputiert und skaliert und die kategorialen imputiert und per One-Hot Encoding kodiert. Wir fassen ihn mit dem Modell in einer `Pipeline` zusammen, sodass jeder Transformer nur auf den Trainings-Folds gefittet wird, auch innerhalb der Kreuzvalidierung. Die Hilfsfunktion unten baut diese Pipeline für einen gegebenen Scaler.

In [ ]:
def build_pipeline(scaler="passthrough", numeric=None):
    if numeric is None:
        numeric = num_cols
    # Imputation vor dem Skalieren in jedem Trainings-Fold fitten.
    numerical = Pipeline(
        [
            ("impute", SimpleImputer(strategy="median", keep_empty_features=True)),
            ("scale", scaler),
        ]
    )
    # Fehlende Kategorien vor dem Kodieren füllen; beide Schritte auf den Trainings-Folds lernen.
    categorical = Pipeline(
        [
            (
                "impute",
                SimpleImputer(strategy="most_frequent", keep_empty_features=True),
            ),
            ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]
    )
    features = ColumnTransformer(
        [
            ("num", numerical, numeric),
            ("cat", categorical, cat_cols),
        ]
    )
    return Pipeline(
        [
            ("features", features),
            ("model", LogisticRegression(max_iter=10000, random_state=RSEED)),
        ]
    )

Für diesen regularisierten Vergleich bleiben alle One-Hot-Indikatoren erhalten.
Mit `handle_unknown="ignore"` wird eine nicht gesehene Kategorie zu einem Block aus lauter Nullen;
er wird nicht mit einer absichtlich weggelassenen Referenzkategorie verwechselt.
In einem echten System sollten solche Eingaben trotzdem untersucht werden.

## Schritte 6 bis 8: Modelltraining und Hyperparameter-Tuning

Feature Engineering, Modelltraining, Hyperparameter-Tuning und Validierung bilden eine iterative Schleife. Wir beginnen mit einer einfachen Baseline und testen dann, ob Skalierung und getunte Hyperparameter sie übertreffen. Eine Verbesserung ist nicht garantiert.

In [ ]:
# In allen Vergleichen gilt Krankheit vorhanden als Klasse 1.
validation_results = {}
class_names = ["No disease recorded", "Disease present"]


def evaluate(name, y_true, y_pred):
    metrics = {
        "F2": fbeta_score(y_true, y_pred, beta=2, pos_label=1, zero_division=0),
        "Precision": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "Recall": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "Accuracy": accuracy_score(y_true, y_pred),
    }
    print(f"{name}: accuracy {metrics['Accuracy']:.2f} | F2 {metrics['F2']:.2f}")
    fig, ax = plt.subplots(figsize=(6, 4))
    sns.heatmap(
        confusion_matrix(y_true, y_pred, labels=[0, 1]),
        annot=True,
        fmt="d",
        cmap="YlGnBu",
        xticklabels=class_names,
        yticklabels=class_names,
        ax=ax,
    )
    ax.set(title=name, xlabel="Predicted status", ylabel="Recorded status")
    fig.tight_layout()
    plt.show()
    return metrics

### Baseline-Modelle

Behalt die ursprüngliche einfache `max_heartrate`-Regel bei: Sag die Klasse vorher, deren Mittelwert der Trainingsdaten näher liegt. Die Grenze ist der Mittelpunkt der beiden Klassenmittelwerte; ihre Richtung wird aus den Labels der Trainingsdaten gelernt, statt anzunehmen, dass eine höhere Herzfrequenz ein höheres Risiko bedeutet. Fehlende Messwerte erhalten den Median der Trainingsdaten. Das ist eine didaktische Heuristik, keine klinische Regel.

Vergleich außerdem einen konstant positiven Klassifikator: Der Recall ist 1, wenn positive Fälle vorhanden sind, aber jeder negative Fall wird zu einem False Positive. Diese zusätzliche Referenz zeigt, warum ein hoher F2-Score allein nicht reicht.

In [ ]:
def predict_heartrate_rule(training, rows):
    """Lernt eine Mittelpunktregel aus den Trainingsdaten und wendet sie auf neue Zeilen an."""
    observed = training["max_heartrate"].dropna()
    if observed.empty:
        raise ValueError("The heart-rate rule needs observed training measurements.")
    # Ersatzwerte und die Klassengrenze nur aus den Beobachtungen der Trainingsdaten lernen.
    fill_value = observed.median()
    class_means = (
        training["max_heartrate"]
        .fillna(fill_value)
        .groupby(training["disease_present"])
        .mean()
    )
    print("Training class mean heart rates:", class_means.round(1).to_dict())
    print("Midpoint threshold:", round(class_means.mean(), 1))
    values = rows["max_heartrate"].fillna(fill_value)
    # Nächster Klassenmittelwert; bei exaktem Gleichstand wird Klasse 0 vorhergesagt.
    return ((values - class_means[1]).abs() < (values - class_means[0]).abs()).astype(
        int
    )


validation_results["Heart-rate rule"] = evaluate(
    "Heart-rate rule (validation)", y_val, predict_heartrate_rule(train_df, X_val)
)

baseline = DummyClassifier(strategy="constant", constant=1)
baseline.fit(X_train, y_train)
validation_results["Constant positive baseline"] = evaluate(
    "Baseline (validation)", y_val, baseline.predict(X_val)
)

### Erste logistische Regression (und eine Datenleck-Falle)

Jetzt ein echtes Modell. Wir trainieren eine logistische Regression über die Pipeline, zuerst ohne `patient_id` und dann mit, um zu zeigen, wie eine Kennung die Zielvariable verraten kann.

In [ ]:
# Numerische Features unskaliert lassen; innerhalb der Pipeline imputieren und kodieren.
log_reg = build_pipeline()
log_reg.fit(X_train, y_train)
validation_results["Unscaled logistic regression"] = evaluate(
    "Unscaled logistic regression (validation)", y_val, log_reg.predict(X_val)
)

In [ ]:
# Die Kennung absichtlich aufnehmen, um einen ungültigen Kandidaten zu zeigen.
log_reg_id = build_pipeline(numeric=num_cols + ["patient_id"])
log_reg_id.fit(X_train_id, y_train)
evaluate(
    "Identifier included: invalid candidate (validation)",
    y_val,
    log_reg_id.predict(X_val_id),
)

Wenn das Hinzufügen von `patient_id` die Leistung stark verbessert, vermute ein Artefakt der Datenerhebung oder der Reihenfolge. Eine ID ist nicht automatisch ein Datenleck, aber eine mit dem Label korrelierte Kennung ohne prädiktive Bedeutung ist kein vertretbares Feature. Dieses absichtlich fehlerhafte Modell wird von der Auswahl ausgeschlossen.

In [ ]:
sns.kdeplot(data=train_df, x="patient_id", hue="disease_present", fill=True);

Untersuch, ob die Labels unterschiedliche ID-Bereiche belegen. Das Muster kann sich ändern, wenn sich die Lehrdatenbank ändert; nimm also keinen festen Schwellenwert an.

In [ ]:
_ = sns.scatterplot(data=train_df, x="patient_id", y="disease_present")

In [ ]:
# Prüfen, wie viele Beobachtungen der Trainingsdaten zu jeder Klasse gehören.
train_df.groupby("disease_present")["patient_id"].count().rename("count")

In [ ]:
# ID-Bereiche der Klassen vergleichen, ohne einen festen Schwellenwert in der Datenbank anzunehmen.
id_ranges = train_df.groupby("disease_present")["patient_id"].agg(["min", "max"])
id_ranges

In [ ]:
# Überlappende Bereiche schließen Artefakte der Reihenfolge nicht aus; auch die Plots verwenden.
overlap_start = id_ranges["min"].max()
overlap_end = id_ranges["max"].min()
print("Class ID ranges overlap:", overlap_start <= overlap_end)

Die Reihenfolge in der Datenbank kann widerspiegeln, wie Einträge importiert oder sortiert wurden. Eine zufällige Aufteilung der Zeilen kann solche Artefakte in jeder Teilmenge erhalten und irreführend hohe Scores liefern. Diese Lösung verwendet keinen festen Schwellenwert für die Patienten-ID.

### Training mit Feature-Skalierung

Vergleich die Standardisierung mit der Min-Max-Skalierung auf demselben Validierungsdatensatz. Nur der Scaler ändert sich; die Vorverarbeitung und die Modellwahl bleiben ansonsten gleich.

#### Standardisierung

In [ ]:
std_pipe = build_pipeline(StandardScaler())
std_pipe.fit(X_train, y_train)
validation_results["Standardized logistic regression"] = evaluate(
    "Standardized (validation)", y_val, std_pipe.predict(X_val)
)

#### Min-Max-Skalierung

In [ ]:
minmax_pipe = build_pipeline(MinMaxScaler())
minmax_pipe.fit(X_train, y_train)
validation_results["Min-max logistic regression"] = evaluate(
    "Min-max (validation)", y_val, minmax_pipe.predict(X_val)
)

### Hyperparameter-Tuning mit RandomizedSearchCV

Führ die Suche über die vollständige standardisierte Pipeline mit stratifizierten Trainings-Folds durch. Halt das Budget überschaubar: zehn gezogene Kombinationen und fünf Folds. Variiere `C` (kleiner bedeutet stärkere Regularisierung) und `l1_ratio` (0 = L2, 1 = L1, Werte dazwischen = Elastic Net). Nutz `saga`, das alle drei unterstützt.

L1 kann Koeffizienten auf null setzen; L2 verkleinert Koeffizienten, ohne sie in der Regel exakt auf null zu setzen. Das Random Search zieht Mischungen; das Netz unten vergleicht ausdrücklich die beiden Endpunkte.

[Referenz: scikit-learn LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html).

Das Präfix `model__` spricht Parameter des Pipeline-Schritts `model` an; die Vorverarbeitung bleibt in jedem Fold, auf dem trainiert wird.

In [ ]:
# F2 gewichtet den Recall stärker; auch Precision und Fehleranzahlen untersuchen.
f2_scorer = make_scorer(fbeta_score, beta=2, pos_label=1, zero_division=0)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RSEED)
if y_train.value_counts().min() < cv.n_splits:
    raise ValueError("Need at least five training examples per class for five-fold CV.")

random_search = RandomizedSearchCV(
    build_pipeline(StandardScaler()),
    {
        "model__solver": ["saga"],
        "model__C": loguniform(0.01, 10),
        "model__l1_ratio": uniform(0, 1),
    },
    n_iter=10,
    scoring=f2_scorer,
    cv=cv,
    n_jobs=-1,
    random_state=RSEED,
    verbose=1,
    error_score="raise",
)
random_search.fit(X_train, y_train)
print("Best parameters:", random_search.best_params_)
print(f"Best training CV F2: {random_search.best_score_:.3f}")

### Hyperparameter-Tuning mit GridSearchCV

Evaluiere vier C-Werte jeweils für L2 (`l1_ratio=0`) und L1 (`l1_ratio=1`): acht Kombinationen, mit denselben Folds und derselben Metrik wie beim Random Search. Vergleich die **CV**-Ergebnisse, um die Tuning-Methode auszuwählen, bevor du den Validierungsdatensatz für den abschließenden Vergleich der Kandidaten nutzt. Keine der beiden Suchen garantiert das bestmögliche Modell außerhalb ihres Suchraums.

Diese C-Werte liegen gleichmäßig verteilt auf einer logarithmischen Skala, entsprechend `np.logspace(-2, 1, num=4)`.

In [ ]:
# L2 und L1 mit denselben Folds und derselben Metrik wie beim Randomized Search vergleichen.
grid_search = GridSearchCV(
    build_pipeline(StandardScaler()),
    {
        "model__solver": ["saga"],
        "model__C": [0.01, 0.1, 1, 10],
        "model__l1_ratio": [0, 1],
    },
    scoring=f2_scorer,
    cv=cv,
    n_jobs=-1,
    verbose=1,
    error_score="raise",
)
grid_search.fit(X_train, y_train)
print("Best parameters:", grid_search.best_params_)
print(f"Best training CV F2: {grid_search.best_score_:.3f}")

selected_search = max(
    [grid_search, random_search], key=lambda search: search.best_score_
)
tuned_model = selected_search.best_estimator_
validation_results["Tuned logistic regression"] = evaluate(
    "Tuned model (validation)", y_val, tuned_model.predict(X_val)
)
# Infrage kommende Kandidaten auf denselben Validierungszeilen vergleichen; Testergebnisse getrennt halten.
validation_comparison = pd.DataFrame.from_dict(validation_results, orient="index")
validation_comparison.sort_values("F2", ascending=False, kind="stable")

## Schritt 9: Test-Score berechnen

Wähl unter den infrage kommenden Kandidaten, beide Baselines eingeschlossen, den mit dem höchsten F2-Score auf den Validierungsdaten. Bei Gleichstand entscheidet die Reihenfolge, in der sie in `validation_results` eingefügt wurden. Trainiere das ausgewählte Modell auf Trainings- und Validierungsdaten neu (oder lern die Herzfrequenz-Regel neu) und evaluiere dann **einmal** auf dem Testdatensatz.

Die Suchobjekte wurden bereits auf ihren Trainingseingaben neu trainiert, haben aber die separate Validierungsteilmenge nicht gesehen. Gib die abschließende Confusion Matrix, Precision, Recall und F2 an. Wenn die Baseline gewinnt, berichte dieses Ergebnis, statt zu behaupten, das komplexere Modell sei besser.

Lies Precision und Recall zusammen mit F2: Die konstant positive Baseline hat einen Recall von 1, aber ihre Precision entspricht dem Anteil der positiven Klasse. F2 bevorzugt den Recall; ein hoher F2-Score allein bedeutet also noch kein nützliches Modell. Die Metriken beziehen sich auf Klasse 1; nicht definierte Verhältnisse werden per Konvention als 0 angegeben. Die Tabelle ist nur für die Anzeige gerundet; für die Auswahl wird der ungerundete F2-Score genutzt.

In [ ]:
candidates = {
    "Constant positive baseline": baseline,
    "Unscaled logistic regression": log_reg,
    "Standardized logistic regression": std_pipe,
    "Min-max logistic regression": minmax_pipe,
    "Tuned logistic regression": tuned_model,
}
selected_name = max(validation_results, key=lambda name: validation_results[name]["F2"])
X_test, y_test = test_df[feature_cols], test_df["disease_present"]
if selected_name == "Heart-rate rule":
    y_pred = predict_heartrate_rule(development_df, X_test)
else:
    # Die ausgewählte Konfiguration vor dem abschließenden Test auf den Entwicklungsdaten neu trainieren.
    best_model = clone(candidates[selected_name])
    best_model.fit(development_df[feature_cols], development_df["disease_present"])
    y_pred = best_model.predict(X_test)
print(f"Selected using validation data: {selected_name}")
test_metrics = evaluate("Final test", y_test, y_pred)
print(
    classification_report(
        y_test, y_pred, labels=[0, 1], target_names=class_names, zero_division=0
    )
)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()
print(f"Detected disease records: {tp}; missed disease records: {fn}.")
print(f"False positives: {fp}; correctly negative records: {tn}.")

## Schritt 10: Deployment und Grenzen

Die Datenbank bietet weiterhin eine nützliche Aufgabe zu SQL und Datenqualität, aber dieses Modell
darf **nicht** als Tool für medizinische Entscheidungen bereitgestellt werden.
Die Quelle ist eine kleine historische diagnostische Stichprobe, keine repräsentative
Grundgesamtheit. Die Lehrfassung fügt fehlende Messwerte hinzu, ändert Codes
und enthält ein wiederholtes Profil; ihre korrigierte Zielvariable ist die dokumentierte Erkrankung,
kein künftiges Ereignis.

Eine einzige kleine zufällige Testaufteilung belegt keine klinische Validität, keine kalibrierten
Wahrscheinlichkeiten, keine Fairness und keine Sicherheit. Tune die Hyperparameter nicht erneut, nachdem du die Testfehler angesehen hast.

Leg in einem echten Projekt fest, wann die Vorhersage erfolgt und welche Messwerte
dann vorliegen. Spezielle Tests sind vielleicht nicht verfügbar oder zu teuer.
Hol eine angemessene fachliche Prüfung und eine unabhängige, repräsentative Validierung ein.
Überwache die Eingabequalität, Verschiebungen der Verteilung, die Leistung in Untergruppen und die
Fehlerkosten, nicht nur die Accuracy insgesamt.

## Prüfe dein Verständnis: Antworten

1. **Warum `patient_id` ausschließen?** Die ID identifiziert eine Person, statt einen verallgemeinerbaren Messwert zu liefern. Sie kann die Reihenfolge der Erhebung oder die Herkunft kodieren und eine irreführende Abkürzung schaffen. Behalt sie für Joins und Duplikatprüfungen, nicht als Prädiktor.
2. **Warum die Imputation innerhalb jedes Folds fitten?** Mediane oder andere Ersatzwerte dürfen nur aus den Trainingszeilen dieses Folds stammen. Werden sie vor der Kreuzvalidierung berechnet, beeinflussen zurückgehaltene Zeilen die Vorverarbeitung, und die Evaluation kann verzerrt werden. Die Pipeline setzt diese Grenze durch.
3. **Warum nicht anhand der Test-Scores auswählen?** Die Wahl des besten Test-Scores nutzt den Testdatensatz für die Modellauswahl; dadurch wird der Test-Score zu einer optimistischen Schätzung der Leistung des ausgewählten Modells. Schreib die Entscheidungen mit der CV auf den Trainingsdaten und der Validierung fest, trainiere auf den Entwicklungsdaten neu und evaluiere dann einmal auf dem Testdatensatz.
4. **Warum auch die rohe Spalte `heart_attack` ausschließen?** Sie bestimmt direkt
   `disease_present = 1 - heart_attack`. Als Eingabe würde sie dem
   Klassifikator die Antwort verraten. Behalt sie nur, um die Zuordnung der Zielvariable zu überprüfen.